# Checking ¹H NMR descriptions in SI files (Google Colab)

Tools for checking published ¹H NMR descriptions for internal consistency
(checkHNMR, local version). No installation needed: run the three cells below one after
another (the ▶ button on the left of each cell).

**Privacy.** The files you upload are processed on Google servers. For published SI this does
not matter. If you check data that are not yet published (e.g. before submitting a paper),
use the local version instead (`run_all.bat` on your own computer), where nothing leaves your
computer.

**What the tools are.** A first-pass check of a written description for inconsistencies, not a
spectral analysis. Every flag is a pointer for the chemist to check; the final decision is made
by a person looking at the spectrum. Details: the README files in the repository.

## 1. Install the tools (about a minute)

By default the tools are taken from GitHub. Before the repository is public, set
`SOURCE = "zip"` and upload the zip archive of the tools when asked.

In [ ]:
SOURCE = "github"   # "github" or "zip"
REPO = "https://github.com/ArturEshtukov-Sheglov/checkHNMR"

import os, glob, shutil, subprocess, zipfile
from google.colab import files
os.chdir("/content")
shutil.rmtree("/content/tools", ignore_errors=True)
if SOURCE == "github":
    subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/tools"], check=True)
else:
    print("Choose the zip archive with the tools:")
    up = files.upload()
    with zipfile.ZipFile(next(iter(up))) as z:
        z.extractall("/content/tools")
found = glob.glob("/content/tools/**/run_all.py", recursive=True)
assert found, "run_all.py not found in the tools"
TOOLS = os.path.dirname(found[0])
subprocess.run(["pip", "install", "-q", "pymupdf", "rdkit", "openpyxl", "python-docx"], check=True)
print("Tools ready in", TOOLS)

## 2. Upload the SI files (PDF)

Word files must first be saved as PDF. Several files can be chosen at once.

In [ ]:
data = os.path.join(TOOLS, "data")
os.makedirs(data, exist_ok=True)
for f in glob.glob(os.path.join(data, "*.pdf")):
    os.remove(f)                        # start from an empty data folder
up = files.upload()
for name, content in up.items():
    with open(os.path.join(data, name), "wb") as fh:
        fh.write(content)
print("In data:", sorted(os.listdir(data)))

## 3. Run the check and download the results

About a minute per file. At the end a zip with the tables is downloaded:
one folder per PDF with `compounds_…`, `smiles_…` and `checked_…` (open `checked_…` first),
plus `run_summary.csv` and `run_log.txt`.

In [ ]:
shutil.rmtree(os.path.join(TOOLS, "results"), ignore_errors=True)
!cd "$TOOLS" && python run_all.py
shutil.make_archive("/content/results", "zip", os.path.join(TOOLS, "results"))
files.download("/content/results.zip")